# Inference Tests

In [18]:
import torch
from blissnet.blissnet import BLISSNet
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, random_split
import os
import xarray
from tqdm import tqdm
from dataclasses import dataclass
import pickle
import plotly.express as px

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


## Dataset Initialization

In [19]:
class T2MDataset(Dataset):
    def __init__(self, ds, transform=None):
        super().__init__()
        xarray_ds = ds

        ds_t2m = torch.from_numpy(xarray_ds['t2m'].to_numpy())
        T, H, W = ds_t2m.shape
        ds_t2m = list(ds_t2m.reshape(T, -1).unsqueeze(dim=-1).unbind(dim=0))

        vec_lat = torch.from_numpy(xarray_ds.latitude.to_numpy())
        vec_long = torch.from_numpy(xarray_ds.longitude.to_numpy())
        arr_lat, arr_long = torch.meshgrid([vec_lat, vec_long], indexing='ij')
        grid_coord = list(torch.stack([arr_lat, arr_long], dim=-1).reshape(H*W,2).unsqueeze(0).expand(T, H*W, 2).unbind(dim=0))

        self.H = H
        self.W = W
        self.T = T
        self.dataset = list(zip(ds_t2m, grid_coord))
        ds_t2m_raw = torch.from_numpy(xarray_ds['t2m'].to_numpy()) 
        self.transform = transform
        
    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        if index >= len(self.dataset) or index < 0:
            raise IndexError

        item = self.dataset[index]
        if self.transform is not None:
            item = self.transform(item)

        return item

In [20]:
class DatasetScalar:
    def __init__(self, t2m_min=200.0, t2m_max=340.0,
                 lat_min=-90.0, lat_max=90.0,
                 long_min=-180.0, long_max=180.0):
        self.t2m_min = t2m_min
        self.t2m_max = t2m_max
        self.coord_min = torch.tensor([lat_min, long_min], dtype=torch.float32)
        self.coord_max = torch.tensor([lat_max, long_max], dtype=torch.float32)

    def norm_t2m(self, t2m):
        t2m = 2 * (t2m - self.t2m_min) / (self.t2m_max - self.t2m_min) - 1
        return t2m.clamp(-1, 1)
    
    def denorm_t2m(self, t2m):
        return (t2m + 1) / 2 * (self.t2m_max - self.t2m_min) + self.t2m_min
    
    def norm_coord(self, coord):
        coord_min = self.coord_min.to(coord.device)
        coord_max = self.coord_max.to(coord.device)
        return 2 * (coord - coord_min) / (coord_max - coord_min) - 1
    
    def denorm_coord(self, coord):
        coord_min = self.coord_min.to(coord.device)
        coord_max = self.coord_max.to(coord.device)
        return (coord + 1) / 2 * (coord_max - coord_min) + coord_min

    def norm(self, sample):
        t2m, coord = sample
        return self.norm_t2m(t2m), self.norm_coord(coord)

    def denorm(self, sample):
        t2m, coord = sample
        return self.denorm_t2m(t2m), self.denorm_coord(coord)

    def __call__(self, sample):
        return self.norm(sample)

In [21]:
scalar = DatasetScalar()

In [22]:
dataset_dir = "./datasets"
south_asia_dataset_org_xarr = xarray.open_dataset(f'{dataset_dir}/south_asia_t2m.nc')
punjab_dataset_org_xarr = xarray.open_dataset(f'{dataset_dir}/punjab_t2m.nc')
karnatka_dataset_org_xarr = xarray.open_dataset(f'{dataset_dir}/karnatka_t2m.nc')

In [23]:
dataset_dir = "./datasets"
south_asia_dataset = T2MDataset(south_asia_dataset_org_xarr, transform=scalar)
punjab_dataset = T2MDataset(punjab_dataset_org_xarr, transform=scalar)
karnatka_dataset = T2MDataset(karnatka_dataset_org_xarr, transform=scalar)

## Dataset Characteristics
1. South Asia (T2M) Dataset:
-> Covers whole south asian bounds with diverse climate spatially and temporally
2. Punjab (T2M) Dataset:
-> Spatially consistent but highly varying temperature throughout year
3. Karnatka (T2M) Dataset:
-> Spatially and timely consistent temperature due to it being a tropical zone.

## Inference Experiments

For each model trained on a dataset, we'll perform inference test for each other dataset, to judge and evaluate following characteristics:
1. How less diverse region trained model performs on diverse regions
2. How broader region trained model can perform on smaller region in detailed manner
3. How consistent temperature region trained model performs on highly varying temperature region

In [24]:
class BLISSNetInterface():
    def __init__(self, dataset:T2MDataset, config, device=None):
        self.config = config
        self.device = device
        if self.device is None:
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        self.transform:DatasetScalar = dataset.transform
        self.dataset_params = {'H':dataset.H, 'W':dataset.W, 'T':dataset.T}
        self.model = BLISSNet(config.emb_dim, dataset.H, dataset.W, config.K, config.n_heads, config.dropout).to(self.device)

    def rmse(self, pred, target):
        return torch.sqrt(torch.mean((pred - target) ** 2)).item()

    def mae(self, pred, target):
        return torch.mean(torch.abs(pred - target)).item()

    def bias(self, pred, target):
        return torch.mean(pred - target).item()

    def r2_score(self, pred, target):
        ss_res = torch.sum((target - pred) ** 2)
        ss_tot = torch.sum((target - target.mean()) ** 2)
        return (1 - ss_res / (ss_tot + 1e-8)).item()

    def compute_metrics(self, pred, target):
        pred = pred.float()
        target = target.float()
        return {
            'rmse': self.rmse(pred, target),
            'mae': self.mae(pred, target),
            'bias': self.bias(pred, target),
            'r2_percent': max(0.0, self.r2_score(pred, target)) * 100,
        }
    
    @torch.no_grad()
    def super_resolve(self, multiplier, dataset:T2MDataset, timestep):
        # using indiviual timestep inference because batch T inference was memory-expensive for GPU
        self.model.eval()

        t2m, coords = dataset[timestep]
        H, W = dataset.H, dataset.W

        new_H = H * multiplier
        new_W = W * multiplier

        t2m = t2m.unsqueeze(0).to(self.device).float()  # (1, H*W, 1)

        coords = coords.unsqueeze(0)
        coords = coords.to(self.device).float()        # (1, H*W, 2)
        bounds = (coords[..., 0].min(), coords[..., 0].max(), coords[..., 1].min(), coords[..., 1].max())
        lat = torch.linspace(
            coords[..., 0].min(),
            coords[..., 0].max(),
            new_H,
            device=self.device,
        )

        lon = torch.linspace(
            coords[..., 1].min(),
            coords[..., 1].max(),
            new_W,
            device=self.device,
        )

        lat_grid, lon_grid = torch.meshgrid(lat, lon, indexing="ij")
        new_coords = torch.stack((lat_grid, lon_grid), dim=-1).reshape(
            1, new_H, new_W, 2
        )

        new_t2m, _, _ = self.model((t2m, coords), (new_H, new_W), bounds, phase=1)
        new_t2m = new_t2m.reshape(new_H, new_W)

        metrics = None
        if new_H == H and new_W == W:
            target = t2m.reshape(H, W)
            denorm_t2m = self.transform.denorm_t2m(new_t2m)
            denorm_target = self.transform.denorm_t2m(target)
            metrics = self.compute_metrics(denorm_t2m, denorm_target)
        else:
            native_t2m, _, _ = self.model((t2m, coords), (H, W), bounds, phase=1)
            native_t2m = native_t2m.reshape(H, W)
            target = t2m.reshape(H, W)

            denorm_t2m = self.transform.denorm_t2m(native_t2m)
            denorm_target = self.transform.denorm_t2m(target)
            metrics = self.compute_metrics(denorm_t2m, denorm_target)

        new_t2m, new_coords = self.transform.denorm((new_t2m.to('cpu'), new_coords.to('cpu')))

        return new_t2m.reshape(1, new_H, new_W, 1), new_coords, metrics


    def load(self, name):
        if not os.path.exists(f"./models/{name}_blissnet_chkpoint.pth"):
            return False

        self.model.setPhase(0, self.config)
        self.model.setPhase(1, self.config)
        self.model.to(self.device)

        chkpoint = torch.load(f'./models/{name}_blissnet_chkpoint.pth', map_location=self.device)
        self.model.load_state_dict(chkpoint)
        return True

In [25]:
@dataclass
class Configuration():
    batch_size: int
    epochs:int
    lr: float
    emb_dim: int
    K: int
    n_heads: int
    dropout: float
    in_channels: int
    base_channels: int
    n_groups: int
    n_transformer_layers: int
    n_hidden_linear_layers: int
    siren_hidden_dim: int
    siren_layers: int
    omega: int

In [26]:
config = Configuration(
    batch_size=32,
    epochs=15,                 
    lr=1e-5,
    emb_dim=512,
    K=512,
    n_heads=8,
    dropout=0.1,
    in_channels=1,
    base_channels=64,
    n_groups=8,
    n_transformer_layers=4,
    n_hidden_linear_layers=3,
    siren_hidden_dim=512,
    siren_layers=4,
    omega=30
)

In [27]:
def average_metrics(metrics_list):
    keys = metrics_list[0].keys()
    return {k: sum(m[k] for m in metrics_list) / len(metrics_list) for k in keys}

In [28]:
def get_xarray(t2m, coords, time_vals=None):
    t2m = t2m.detach().cpu().numpy() if hasattr(t2m, 'detach') else t2m
    coords = coords.detach().cpu().numpy() if hasattr(coords, 'detach') else coords

    T, H, W, _ = t2m.shape

    lat = coords[0, :, 0, 0].astype(np.float64)
    lon = coords[0, 0, :, 1].astype(np.float64)

    if time_vals is None:
        time_vals = np.arange('2025-01-01T00:00:00', '2026-01-01T00:00:00', np.timedelta64(1, 'D'), dtype='datetime64[ns]')[:T]

    ds = xarray.Dataset(
        {
            "t2m": (["time", "latitude", "longitude"], t2m.squeeze(-1).astype(np.float32))
        },
        coords={
            "time": time_vals,
            "latitude": lat,
            "longitude": lon,
        }
    )
    return ds

In [29]:
def visualize(ds:xarray.Dataset, title=""):
    selected_periods = [
        ("2025-01-15", "2025-01-19"), 
        ("2025-04-15", "2025-04-19"),  
        ("2025-07-15", "2025-07-19"),  
        ("2025-10-15", "2025-10-19"), 
    ]

    ds_vis = xarray.concat(
        [
            ds.sel(time=slice(start, end))
            for start, end in selected_periods
        ],
        dim="time"
    )

    ds_vis = ds_vis.copy()
    ds_vis['t2m'] = ds_vis['t2m'] - 273.15
    t_min = ds_vis['t2m'].min().item()
    t_max = ds_vis['t2m'].max().item()
    fig = px.imshow(
        ds_vis['t2m'], 
        animation_frame='time', 
        x=ds_vis['longitude'], 
        y=ds_vis['latitude'],   
        color_continuous_scale='RdBu_r',
        title="t2m Heatmap over Time",
        height=1000,
        width=1000,
        range_color=[t_min, t_max]
    )

    fig.write_html(f"./outputs/{title}_4x.html")
    print("Figure saved.")
    return fig

In [30]:
os.makedirs('./outputs', exist_ok=True)

In [31]:
def inference(dataset:xarray.Dataset, blissnet:BLISSNetInterface, title=""):
    all_metrics = []
    T_t2m = []
    T_coords = []
    for timestep in tqdm(range(0, dataset.T, 4), "Super-resolving"):
        t2m, coords, metrics = blissnet.super_resolve(4, dataset, timestep)
        T_t2m.append(t2m)
        T_coords.append(coords)
        all_metrics.append(metrics)

    T_t2m = torch.cat(T_t2m, dim=0)
    T_coords = torch.cat(T_coords, dim=0)

    xr_ds = get_xarray(T_t2m, T_coords)
    avg_metrics = average_metrics(all_metrics)

    xr_ds.to_netcdf(f'./outputs/{title}_4x.nc')
    with open(f'./outputs/{title}_4x_metrics.pkl', 'wb') as file:
        pickle.dump(avg_metrics, file)

    print("Saved.")
    return xr_ds, avg_metrics

# Experiments

## Experiment 1 : South Asia Trained Model

In [32]:
south_asia_blissnet = BLISSNetInterface(south_asia_dataset, config, device)
if south_asia_blissnet.load('south_asia'):
    print("Checkpoint Loaded.")

C:\Users\Syed Daniyal\AppData\Local\Temp\ipykernel_8640\4125734134.py:102: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  chkpoint = torch.load(f'./models/{name}_blissnet_chk

Checkpoint Loaded.


### 1 (a): Against South Asia

In [33]:
xr_ds, avg_metrics = inference(south_asia_dataset, south_asia_blissnet, 'sa_sa')

Super-resolving: 100%|██████████| 365/365 [05:56<00:00,  1.02it/s]


Saved.


In [34]:
xr_ds = xarray.open_dataset('./outputs/sa_sa_4x.nc')
with open('./outputs/sa_sa_4x_metrics.pkl', 'rb') as file:
    avg_metrics = pickle.load(file)

print('Loaded.')

Loaded.


In [35]:
fig = visualize(xr_ds, 'sa_sa')

Figure saved.


In [36]:
print("========= METRICS =========")
for k,v in avg_metrics.items():
    print(f"{k.upper()}: {v:.4f}\n")

========= METRICS =========
RMSE: 2.3298

MAE: 1.6486

BIAS: -0.2378

R2_PERCENT: 95.8292



### 1 (b): Against Punjab

In [37]:
xr_ds, avg_metrics = inference(punjab_dataset, south_asia_blissnet, 'sa_pj')

Super-resolving: 100%|██████████| 365/365 [00:08<00:00, 40.76it/s]


Saved.


In [38]:
xr_ds = xarray.open_dataset('./outputs/sa_pj_4x.nc')
with open('./outputs/sa_pj_4x_metrics.pkl', 'rb') as file:
    avg_metrics = pickle.load(file)

print('Loaded.')

Loaded.


In [39]:
fig = visualize(xr_ds, 'sa_pj')

Figure saved.


In [40]:
print("========= METRICS =========")
for k,v in avg_metrics.items():
    print(f"{k.upper()}: {v:.4f}\n")

========= METRICS =========
RMSE: 8.1258

MAE: 7.1583

BIAS: 5.1340

R2_PERCENT: 2.4286



### 1 (c): Against Karnatka

In [41]:
xr_ds, avg_metrics = inference(karnatka_dataset, south_asia_blissnet, 'sa_kt')

Super-resolving: 100%|██████████| 365/365 [00:07<00:00, 48.23it/s]

Saved.


In [42]:
xr_ds = xarray.open_dataset('./outputs/sa_kt_4x.nc')
with open('./outputs/sa_kt_4x_metrics.pkl', 'rb') as file:
    avg_metrics = pickle.load(file)

print('Loaded.')

Loaded.


In [43]:
fig = visualize(xr_ds, 'sa_kt')

Figure saved.


In [44]:
print("========= METRICS =========")
for k,v in avg_metrics.items():
    print(f"{k.upper()}: {v:.4f}\n")

========= METRICS =========
RMSE: 3.2671

MAE: 2.6370

BIAS: 1.5217

R2_PERCENT: 0.0000



## Experiment 2 : Punjab Trained Model

In [45]:
punjab_blissnet = BLISSNetInterface(punjab_dataset, config, device)
if punjab_blissnet.load('punjab'):
    print("Checkpoint Loaded.")

Checkpoint Loaded.


C:\Users\Syed Daniyal\AppData\Local\Temp\ipykernel_8640\4125734134.py:102: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  chkpoint = torch.load(f'./models/{name}_blissnet_chk

### 1 (a): Against South Asia

In [46]:
xr_ds, avg_metrics = inference(south_asia_dataset, punjab_blissnet, 'pj_sa')

Super-resolving: 100%|██████████| 365/365 [06:26<00:00,  1.06s/it]


Saved.


In [47]:
xr_ds = xarray.open_dataset('./outputs/pj_sa_4x.nc')
with open('./outputs/pj_sa_4x_metrics.pkl', 'rb') as file:
    avg_metrics = pickle.load(file)

print('Loaded.')

Loaded.


In [48]:
fig = visualize(xr_ds, 'pj_sa')

Figure saved.


In [49]:
print("========= METRICS =========")
for k,v in avg_metrics.items():
    print(f"{k.upper()}: {v:.4f}\n")

========= METRICS =========
RMSE: 25.5198

MAE: 22.1951

BIAS: -21.3249

R2_PERCENT: 0.0000



### 1 (b): Against Punjab

In [50]:
xr_ds, avg_metrics = inference(punjab_dataset, punjab_blissnet, 'pj_pj')

Super-resolving: 100%|██████████| 365/365 [00:07<00:00, 46.39it/s]


Saved.


In [51]:
xr_ds = xarray.open_dataset('./outputs/pj_pj_4x.nc')
with open('./outputs/pj_pj_4x_metrics.pkl', 'rb') as file:
    avg_metrics = pickle.load(file)

print('Loaded.')

Loaded.


In [52]:
fig = visualize(xr_ds, 'pj_pj')

Figure saved.


In [53]:
print("========= METRICS =========")
for k,v in avg_metrics.items():
    print(f"{k.upper()}: {v:.4f}\n")

========= METRICS =========
RMSE: 1.7759

MAE: 1.3483

BIAS: 0.6318

R2_PERCENT: 82.2801



### 1 (c): Against Karnatka

In [54]:
xr_ds, avg_metrics = inference(karnatka_dataset, punjab_blissnet, 'pj_kt')

Super-resolving: 100%|██████████| 365/365 [00:06<00:00, 59.44it/s]

Saved.


In [55]:
xr_ds = xarray.open_dataset('./outputs/pj_kt_4x.nc')
with open('./outputs/pj_kt_4x_metrics.pkl', 'rb') as file:
    avg_metrics = pickle.load(file)

print('Loaded.')

Loaded.


In [56]:
fig = visualize(xr_ds, 'pj_kt')

Figure saved.


In [57]:
print("========= METRICS =========")
for k,v in avg_metrics.items():
    print(f"{k.upper()}: {v:.4f}\n")

========= METRICS =========
RMSE: 35.9439

MAE: 35.7755

BIAS: -35.7755

R2_PERCENT: 0.0000



## Experiment 3 : Karnatka Trained Model

In [58]:
karnatka_blissnet = BLISSNetInterface(karnatka_dataset, config, device)
if karnatka_blissnet.load('karnatka'):
    print("Checkpoint Loaded.")

Checkpoint Loaded.


C:\Users\Syed Daniyal\AppData\Local\Temp\ipykernel_8640\4125734134.py:102: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  chkpoint = torch.load(f'./models/{name}_blissnet_chk

### 1 (a): Against South Asia

In [59]:
xr_ds, avg_metrics = inference(south_asia_dataset, karnatka_blissnet, 'kt_sa')

Super-resolving: 100%|██████████| 365/365 [05:28<00:00,  1.11it/s]


Saved.


In [60]:
xr_ds = xarray.open_dataset('./outputs/kt_sa_4x.nc')
with open('./outputs/kt_sa_4x_metrics.pkl', 'rb') as file:
    avg_metrics = pickle.load(file)

print('Loaded.')

Loaded.


In [61]:
fig = visualize(xr_ds, 'kt_sa')

Figure saved.


In [62]:
print("========= METRICS =========")
for k,v in avg_metrics.items():
    print(f"{k.upper()}: {v:.4f}\n")

========= METRICS =========
RMSE: 19.0956

MAE: 16.7205

BIAS: -11.6284

R2_PERCENT: 0.0000



### 1 (b): Against Punjab

In [63]:
xr_ds, avg_metrics = inference(punjab_dataset, karnatka_blissnet, 'kt_pj')

Super-resolving: 100%|██████████| 365/365 [00:07<00:00, 46.75it/s]


Saved.


In [64]:
xr_ds = xarray.open_dataset('./outputs/kt_pj_4x.nc')
with open('./outputs/kt_pj_4x_metrics.pkl', 'rb') as file:
    avg_metrics = pickle.load(file)

print('Loaded.')

Loaded.


In [65]:
fig = visualize(xr_ds, 'kt_pj')

Figure saved.


In [66]:
print("========= METRICS =========")
for k,v in avg_metrics.items():
    print(f"{k.upper()}: {v:.4f}\n")

========= METRICS =========
RMSE: 19.9174

MAE: 18.3529

BIAS: -17.4178

R2_PERCENT: 0.0000



### 1 (c): Against Karnatka

In [67]:
xr_ds, avg_metrics = inference(karnatka_dataset, karnatka_blissnet, 'kt_kt')

Super-resolving: 100%|██████████| 365/365 [00:06<00:00, 57.25it/s]

Saved.


In [68]:
xr_ds = xarray.open_dataset('./outputs/kt_kt_4x.nc')
with open('./outputs/kt_kt_4x_metrics.pkl', 'rb') as file:
    avg_metrics = pickle.load(file)

print('Loaded.')

Loaded.


In [69]:
fig = visualize(xr_ds, 'kt_kt')

Figure saved.


In [70]:
print("========= METRICS =========")
for k,v in avg_metrics.items():
    print(f"{k.upper()}: {v:.4f}\n")

========= METRICS =========
RMSE: 0.9298

MAE: 0.7491

BIAS: 0.1963

R2_PERCENT: 70.0844



# Results and Evaluation

<table>
<tr>
    <th rowspan="2">Model</th>
    <th colspan="4">South Asia</th>
    <th colspan="4">Punjab</th>
    <th colspan="4">Karnataka</th>
</tr>
<tr>
    <th>RMSE</th><th>MAE</th><th>R²</th><th>Bias</th>
    <th>RMSE</th><th>MAE</th><th>R²</th><th>Bias</th>
    <th>RMSE</th><th>MAE</th><th>R²</th><th>Bias</th>
</tr>
<tr>
    <td>South Asia Model</td>
    <td>2.3298</td><td>1.6486</td><td>95.8292</td><td>-0.2378</td>
    <td>8.1258</td><td>7.1583</td><td>2.4286</td><td>5.1340</td>
    <td>3.2671</td><td>2.6370</td><td>0.0000</td><td>1.5217</td>
</tr>
<tr>
    <td>Punjab Model</td>
    <td>25.5198</td><td>22.1951</td><td>0.0000</td><td>-21.3249</td>
    <td>1.7759</td><td>1.3483</td><td>82.2801</td><td>0.6318</td>
    <td>35.9439</td><td>35.7755</td><td>0.0000</td><td>-35.7755</td>
</tr>
<tr>
    <td>Karnataka Model</td>
    <td>19.0956</td><td>16.7205</td><td>0.0000</td><td>-11.6284</td>
    <td>19.9174</td><td>18.3529</td><td>0.0000</td><td>-17.4178</td>
    <td>0.9298</td><td>0.7491</td><td>70.0844</td><td>0.1963</td>
</tr>
</table>